# FINAL: all statistics, figures and paper numbers (Colab)
Run after the live tests (Phase 7). Upload `mycobot_results.zip` made on the Jetson with
`cd ~/files/ins && zip -qr mycobot_results.zip mycobot_framework -x "mycobot_framework/deploy/views/*/v*.jpg"`.
Produces: offline + live statistics (Wilcoxon, Holm), all paper figures (fig1-fig9), the novelty claims table,
the overview and policy figures from your images, the paper macros, and one zip to download.

In [ ]:
# 1) Upload and unpack
import glob, os, zipfile, sys, subprocess
if not glob.glob('/content/*.zip'):
    from google.colab import files
    files.upload()
z = sorted(glob.glob('/content/*.zip'), key=os.path.getsize)[-1]
zipfile.ZipFile(z).extractall('/content/fw')
ROOT = os.path.dirname(os.path.dirname(sorted(glob.glob('/content/fw/**/deploy/locked_pose.json', recursive=True), key=len)[0]))
os.chdir(ROOT); sys.path.insert(0, ROOT); print('framework folder:', ROOT)
for f in ['data/view_library.npz', 'deploy/lattice_views.json', 'deploy/view_policy_attn.npz', 'deploy/view_policy_mlp.npz']:
    print('  %-32s %s' % (f, 'ok' if os.path.exists(f) else 'MISSING'))
print('  training runs :', len(glob.glob('results_views/train_*/test_summary.json')))
print('  live runs     :', len(glob.glob('hw_logs/viewplan/*/episodes.csv')))
print('  coverage runs :', len(glob.glob('hw_logs/refcover/*/views.csv')))
print('  defect labels :', os.path.exists('results_views/defect_labels.csv'))

In [ ]:
# 2) Statistics, figures, claims, paper macros (each step prints what it did or why it skipped)
def run(*args):
    print('\n$ python', ' '.join(args))
    r = subprocess.run([sys.executable] + list(args), capture_output=True, text=True, cwd=ROOT)
    print((r.stdout + r.stderr)[-3000:])
run('analyze_views.py')
run('plot_all.py')
run('novelty_evidence.py', '--ik-nodes', '250')
run('paper_figures.py', '--out', 'paper/figs')
run('make_paper.py', '--no-pdf')

In [ ]:
# 3) Show the results
from IPython.display import Image, display, Markdown
for f in ['results_views/offline_summary.txt', 'results_views/live_tests.txt', 'results_views/library_stats.txt',
          'results_novelty/claims_table.md', 'hw_logs/view_policy_timing.json', 'results_views/defect_results.json']:
    if os.path.exists(f):
        display(Markdown('**%s**\n```\n%s\n```' % (f, open(f).read()[:6000])))
for f in sorted(glob.glob('paper/figs/*.png')) + sorted(glob.glob('results_novelty/*.png')):
    display(Markdown('**' + f + '**')); display(Image(f, width=900))

In [ ]:
# 4) Download everything for the paper
import shutil
os.makedirs('/content/final_results', exist_ok=True)
for d in ['paper', 'results_views', 'results_novelty']:
    if os.path.exists(d):
        shutil.copytree(d, '/content/final_results/' + d, dirs_exist_ok=True)
for f in ['hw_logs/view_policy_timing.json']:
    if os.path.exists(f):
        shutil.copy(f, '/content/final_results/')
shutil.make_archive('/content/final_results', 'zip', '/content/final_results')
from google.colab import files
files.download('/content/final_results.zip')